# ARIMA on MSFT — scratch notebook

Validation notebook for the ARIMA blog post. Pulls MSFT data, runs the ADF test, plots ACF/PACF, fits ARIMA via `auto_arima`, and forecasts.

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima

In [ ]:
data = yf.Ticker('MSFT').history(period="2y", interval="1d")
close_prices = data.loc[:, 'Close'].dropna()

plt.figure(figsize=(12,4))
plt.plot(close_prices)
plt.title('MSFT Daily Closing Price')
plt.xlabel('Time')
plt.ylabel('Price ($)')
plt.show()

In [ ]:
result = adfuller(close_prices)
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")

In [ ]:
returns = close_prices.pct_change().dropna()

plt.figure(figsize=(12,4))
plt.plot(returns)
plt.title('MSFT Daily Returns')
plt.xlabel('Time')
plt.ylabel('Return')
plt.show()

result2 = adfuller(returns)
print(f"ADF Statistic: {result2[0]:.4f}")
print(f"p-value: {result2[1]:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(returns, lags=30, ax=axes[0])
axes[0].set_title('ACF of MSFT Daily Returns')
plot_pacf(returns, lags=30, ax=axes[1])
axes[1].set_title('PACF of MSFT Daily Returns')
plt.tight_layout()
plt.show()

In [ ]:
model = auto_arima(close_prices, start_p=0, start_q=0, max_p=5, max_q=5, d=1,
                    seasonal=False, trace=True, suppress_warnings=True)

In [ ]:
model = ARIMA(close_prices, order=(0, 1, 0))
fit = model.fit()

forecast = fit.forecast(steps=30)

last_date = close_prices.index[-1]
forecast_index = pd.bdate_range(start=last_date, periods=31)[1:]
forecast = pd.Series(forecast.values, index=forecast_index)

plt.figure(figsize=(12,4))
plt.plot(close_prices[-100:], label='Actual')
plt.plot(forecast, label='Forecast')
plt.legend()
plt.title('MSFT ARIMA Forecast vs Actual')
plt.show()